# Phase 3 — Annotation

Sample ~100 passages, label them using the annotation guide, lock a 20-passage test set.



In [1]:
import json
import random
from pathlib import Path
from textwrap import wrap

import pandas as pd
from sklearn.metrics import cohen_kappa_score

In [2]:
ROOT       = Path('..').resolve()
PASSAGES   = ROOT / 'data' / 'raw' / 'extracted' / 'passages.jsonl'
LABELED_DIR = ROOT / 'data' / 'raw' / 'labeled'
LABELED_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_BRAND = 17   # ~17 × 6 = ~100 total
TEST_SET_N       = 20
RANDOM_SEED      = 42

## Step 1 — Establish annotation_guide.md FIRST


In [ ]:
guide = """# Annotation Guide — green-claims-nlp

**Version:** 1.0  
**Author:** [your name]  
**Date:** [date]

---

## Binary labels

| Label | Name | Criteria |
|---|---|---|
| **0** | Substantive | Specific measurable goals with timelines; third-party verified claims (SBTi, GRI, Deloitte, ZDHC); concrete emissions data with methodology disclosed; named certification standards (not proprietary) |
| **1** | Greenwashing-risk | Vague commitments without quantified targets; unverifiable claims; self-defined proprietary standards without third-party verification; unregulated beauty terms as sustainability claims; relative comparisons without benchmarks; future-tense aspirations without timelines |
| **2** | Ambiguous | Genuine uncertainty — cannot reliably assign 0 or 1 (excluded from model training; tracked separately) |

---

## Edge cases

| Pattern | Rule |
|---|---|
| Self-defined standards ("lower-impact", "sustainably sourced", "responsibly made") | Label 1 unless linked to a named third-party standard |
| Relative claims without external benchmarks ("reduced by X% vs our own 2019 baseline") | Label 0 if methodology disclosed; label 1 if baseline is undefined |
| Future-tense commitments | Label 0 if quantified with timeline; label 1 if aspirational only |
| Own-operations vs supply-chain claims | Same criteria apply — note distinction in the `scope` metadata field |
| Proprietary program membership (Better Cotton Initiative, Fashion Pact) | Label 1 unless specific outcome metrics reported |
| Unregulated beauty terms (*clean*, *natural*, *non-toxic*, *reef-safe*) | Label 1 for beauty industry; note in `notes` field |

---

## Worked examples

### Label 0 — Substantive
1. "Scope 1 and 2 emissions reduced 88% vs 2018 baseline, validated by SBTi."
2. "97.1% MRSL compliance for chemical inputs, measured via ZDHC Gateway."
3. "100% of cotton sourced via Better Cotton Initiative, with BCI-verified outcome metrics for 2024."
4. "GRI 305-1 disclosure: 12,450 tCO2e scope 1 emissions, third-party assured by Deloitte."
5. "SBTi-aligned 1.5°C target: absolute scope 1+2 reduction of 50% by 2030 vs 2019 base year."

### Label 1 — Greenwashing-risk
1. "We are committed to decarbonising our supply chain." (no timeline, no metric)
2. "Made with lower-impact fibres." (Inditex self-defined, no third-party standard)
3. "Sustainably sourced materials." (H&M self-defined standard)
4. "The positive power of fashion." (aspirational brand language)
5. "Our ambition is to become climate positive by 2040." (no interim milestones or current baselines)

### Label 2 — Ambiguous


---

## Notes
- Record uncertainty using `notes` field; write one sentence explaining any label 2 assignment
- Do not retroactively change labels after test set is locked
"""

guide_path = LABELED_DIR / 'annotation_guide.md'
with open(guide_path, 'w') as f:
    f.write(guide)
print(f"Written: {guide_path}")
print("→ Edit this file before starting annotation.")

Written: /Users/mandy.sun/green-claims-nlp/data/raw/labeled/annotation_guide.md
→ Edit this file before starting annotation.


## Step 2 — Sample passages (stratified by brand)

In [4]:
with open(PASSAGES) as f:
    all_passages = [json.loads(line) for line in f]

# Stratified sample: ~17 per brand
random.seed(RANDOM_SEED)
by_brand: dict[str, list] = {}
for p in all_passages:
    by_brand.setdefault(p['brand'], []).append(p)

sample = []
for brand, ps in by_brand.items():
    n = min(SAMPLE_PER_BRAND, len(ps))
    sampled = random.sample(ps, n)
    sample.extend(sampled)
    print(f"{brand:12s}: {n} sampled from {len(ps):,} available")

print(f"\nTotal sample: {len(sample)} passages")

# Assign annotation IDs
for i, p in enumerate(sample):
    p['annotation_id'] = i

H&M         : 17 sampled from 414 available
Zara        : 17 sampled from 385 available
Patagonia   : 17 sampled from 306 available
L'Oreal     : 17 sampled from 19 available
Sephora     : 17 sampled from 95 available
Lush        : 17 sampled from 60 available

Total sample: 102 passages


## Step 3 — Annotation

The sampled passages are exported to `data/raw/labeled/to_annotate.csv`.  
Fill in the `label` column (0 = substantive, 1 = greenwashing-risk, 2 = ambiguous) and optionally the `notes` column, then re-ran this notebook from Step 4 onward.

In [5]:
ANNOTATE_CSV  = LABELED_DIR / 'to_annotate.csv'
PROGRESS_FILE = LABELED_DIR / 'annotation_progress.jsonl'

# Export sample to CSV for manual annotation (only if not already done)
if not ANNOTATE_CSV.exists():
    export_df = pd.DataFrame([{
        'annotation_id': p['annotation_id'],
        'brand':         p['brand'],
        'industry':      p['industry'],
        'role':          p['role'],
        'text':          p['text'],
        'label':         '',    # fill in: 0 / 1 / 2
        'notes':         '',
    } for p in sample])
    export_df.to_csv(ANNOTATE_CSV, index=False)
    print(f"Exported {len(export_df)} passages → {ANNOTATE_CSV}")
    print("\nNext step: open to_annotate.csv, fill in the 'label' column, save, then re-run from Step 4.")
else:
    print(f"to_annotate.csv already exists — proceeding to load labels.")

# Load annotations if the CSV has labels filled in
labeled_so_far: dict[int, dict] = {}
if ANNOTATE_CSV.exists():
    ann_df = pd.read_csv(ANNOTATE_CSV)
    filled = ann_df[ann_df['label'].notna() & (ann_df['label'].astype(str).str.strip() != '')]
    for _, row in filled.iterrows():
        try:
            label = int(row['label'])
        except (ValueError, TypeError):
            continue
        rec = {
            'annotation_id': int(row['annotation_id']),
            'brand':         row['brand'],
            'industry':      row['industry'],
            'role':          row['role'],
            'text':          row['text'],
            'label':         label,
            'notes':         str(row['notes']) if pd.notna(row['notes']) else '',
        }
        labeled_so_far[rec['annotation_id']] = rec

print(f"\nLabeled so far: {len(labeled_so_far)} / {len(sample)} passages")
if len(labeled_so_far) < len(sample):
    unlabeled = len(sample) - len(labeled_so_far)
    print(f"⚠  {unlabeled} passages still need labels. Fill in to_annotate.csv and re-run.")

to_annotate.csv already exists — proceeding to load labels.

Labeled so far: 102 / 102 passages


## Step 4 — Split and save labeled sets

In [6]:
if not labeled_so_far:
    print("No labels found. Fill in to_annotate.csv first.")
else:
    all_labeled = list(labeled_so_far.values())
    ambiguous   = [p for p in all_labeled if p['label'] == 2]
    binary      = [p for p in all_labeled if p['label'] in (0, 1)]

    print(f"Label 0 (substantive):    {sum(1 for p in binary if p['label'] == 0)}")
    print(f"Label 1 (greenwash-risk): {sum(1 for p in binary if p['label'] == 1)}")
    print(f"Label 2 (ambiguous):      {len(ambiguous)}")

    random.seed(RANDOM_SEED)
    label0 = [p for p in binary if p['label'] == 0]
    label1 = [p for p in binary if p['label'] == 1]

    n0 = round(TEST_SET_N * len(label0) / max(len(binary), 1))
    n1 = TEST_SET_N - n0
    test_set  = random.sample(label0, min(n0, len(label0))) + random.sample(label1, min(n1, len(label1)))
    test_ids  = {p['annotation_id'] for p in test_set}
    train_set = [p for p in binary if p['annotation_id'] not in test_ids]

    print(f"\nTrain: {len(train_set)} | Test: {len(test_set)} | Ambiguous (excluded): {len(ambiguous)}")

    def save_jsonl(records, path):
        with open(path, 'w') as f:
            for r in records:
                f.write(json.dumps(r) + '\n')

    save_jsonl(train_set, LABELED_DIR / 'labeled_passages.jsonl')
    save_jsonl(test_set,  LABELED_DIR / 'test_set.jsonl')
    save_jsonl(ambiguous, LABELED_DIR / 'ambiguous_passages.jsonl')

    print("\nSaved:")
    print(f"  {LABELED_DIR / 'labeled_passages.jsonl'}")
    print(f"  {LABELED_DIR / 'test_set.jsonl'}")
    print(f"  {LABELED_DIR / 'ambiguous_passages.jsonl'}")
    print("\n⚠  Test set is now locked. Do not re-examine or adjust based on model results.")

Label 0 (substantive):    60
Label 1 (greenwash-risk): 12
Label 2 (ambiguous):      30

Train: 52 | Test: 20 | Ambiguous (excluded): 30

Saved:
  /Users/mandy.sun/green-claims-nlp/data/raw/labeled/labeled_passages.jsonl
  /Users/mandy.sun/green-claims-nlp/data/raw/labeled/test_set.jsonl
  /Users/mandy.sun/green-claims-nlp/data/raw/labeled/ambiguous_passages.jsonl

⚠  Test set is now locked. Do not re-examine or adjust based on model results.


## Step 5 — Inter-annotator agreement 


In [7]:
# Provide second annotator labels here: {annotation_id: label}
# Leave empty if no second annotator is available.
second_annotator_labels: dict[int, int] = {
    # e.g. 0: 1, 1: 0, 2: 1, ...
}

if second_annotator_labels:
    shared_ids = sorted(second_annotator_labels.keys())
    y1 = [labeled_so_far[i]['label'] for i in shared_ids if i in labeled_so_far]
    y2 = [second_annotator_labels[i] for i in shared_ids if i in labeled_so_far]
    # Restrict to binary labels for kappa (exclude 2)
    pairs = [(a, b) for a, b in zip(y1, y2) if a != 2 and b != 2]
    if pairs:
        kappa = cohen_kappa_score([a for a, _ in pairs], [b for _, b in pairs])
        print(f"Cohen's kappa (binary labels, n={len(pairs)}): {kappa:.3f}")
        if   kappa >= 0.8: print("  → Almost perfect agreement")
        elif kappa >= 0.6: print("  → Substantial agreement")
        elif kappa >= 0.4: print("  → Moderate agreement")
        else:              print("  → Fair/poor agreement — review annotation guide")
    else:
        print("No shared binary-labeled passages found.")
else:
    print("No second annotator data. Skipping kappa computation.")

No second annotator data. Skipping kappa computation.
